In [1]:
!pip install peft trl datasets -q


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [2]:
# 파인튜닝용 데이터 준비
import json
from collections import Counter

with open('context_conditions.json', encoding='utf-8') as f:
    full_dataset = json.load(f)

# train split, useful 조건만 사용
train_data = [d for d in full_dataset if d['split'] == 'train' and d['useful'] is not None]

print(f"train 샘플 수: {len(train_data)}")
print(f"레이블 분포:")
print(Counter(d['label'] for d in train_data))

train 샘플 수: 4890
레이블 분포:
Counter({'comment': 3508, 'support': 642, 'query': 373, 'deny': 367})


In [3]:
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer
import torch

# 데이터 포맷 변환
SYSTEM_PROMPT = """You are a stance classification expert for social media discussions about rumours.

Classify the stance of the TARGET reply using exactly one of these four labels:

- support: The reply explicitly states the rumour IS true or confirmed.
- deny: The reply explicitly states the rumour IS false or fabricated.
- query: The reply asks for sources, evidence, or verification.
- comment: Everything else. The reply does not directly address whether the rumour is true or false.

Respond with ONLY one word: support, deny, query, or comment. No explanation."""


def format_sample(d):
    text = d['useful']
    text = text.replace("[Source]",  "Rumour post:")
    text = text.replace("[Context]", "Previous reply:")
    text = text.replace("[Target]",  "Reply to classify:")

    user_content = (
        f"Read the following and classify the stance of the 'Reply to classify'.\n\n"
        f"{text}\n\n"
        f"Stance label (support / deny / query / comment):"
    )
    return {
        "text": (
            f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n"
            f"<|im_start|>user\n{user_content}<|im_end|>\n"
            f"<|im_start|>assistant\n{d['label']}<|im_end|>"
        )
    }

formatted = [format_sample(d) for d in train_data]
hf_dataset = Dataset.from_list(formatted)

print(f"학습 데이터 준비 완료: {len(hf_dataset)}개")
print("\n샘플 확인:")
print(formatted[0]['text'][:500])

/opt/jupyter/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


학습 데이터 준비 완료: 4890개

샘플 확인:
<|im_start|>system
You are a stance classification expert for social media discussions about rumours.

Classify the stance of the TARGET reply using exactly one of these four labels:

- support: The reply explicitly states the rumour IS true or confirmed.
- deny: The reply explicitly states the rumour IS false or fabricated.
- query: The reply asks for sources, evidence, or verification.
- comment: Everything else. The reply does not directly address whether the rumour is true or false.

Respond


In [12]:
import os
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer, SFTConfig
import torch

# 데이터 포맷 변환
SYSTEM_PROMPT = """You are a stance classification expert for social media discussions about rumours.

Classify the stance of the TARGET reply using exactly one of these four labels:

- support: The reply explicitly states the rumour IS true or confirmed.
- deny: The reply explicitly states the rumour IS false or fabricated.
- query: The reply asks for sources, evidence, or verification.
- comment: Everything else. The reply does not directly address whether the rumour is true or false.

Respond with ONLY one word: support, deny, query, or comment. No explanation."""


def format_sample(d):
    text = d['useful']
    text = text.replace("[Source]",  "Rumour post:")
    text = text.replace("[Context]", "Previous reply:")
    text = text.replace("[Target]",  "Reply to classify:")
    user_content = (
        f"Read the following and classify the stance of the 'Reply to classify'.\n\n"
        f"{text}\n\n"
        f"Stance label (support / deny / query / comment):"
    )
    return {
        "text": (
            f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n"
            f"<|im_start|>user\n{user_content}<|im_end|>\n"
            f"<|im_start|>assistant\n{d['label']}<|im_end|>"
        )
    }


formatted  = [format_sample(d) for d in train_data]
hf_dataset = Dataset.from_list(formatted)
print(f"학습 데이터 준비 완료: {len(hf_dataset)}개")

# 0.5B 모델 로드
MODEL_NAME   = "Qwen/Qwen2.5-0.5B-Instruct"
tokenizer_ft = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer_ft.pad_token = tokenizer_ft.eos_token
model_ft = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto"
)

# LoRA 설정
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)
model_ft = get_peft_model(model_ft, lora_config)
model_ft.print_trainable_parameters()

# 파인튜닝
sft_config = SFTConfig(
    output_dir=os.path.expanduser("~/qwen_0.5b_ft"),
    num_train_epochs=3,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=50,
    save_strategy="epoch",
    warmup_steps=100,
    lr_scheduler_type="cosine",
    report_to="none",
)

trainer = SFTTrainer(
    model=model_ft,
    args=sft_config,
    train_dataset=hf_dataset,
    processing_class=tokenizer_ft,
)

print("0.5B 파인튜닝 시작...")
trainer.train()
print("파인튜닝 완료!")

# 모델 저장
save_path = os.path.expanduser("~/qwen_0.5b_ft/final")
model_ft.save_pretrained(save_path)
tokenizer_ft.save_pretrained(save_path)
print(f"저장 완료: {save_path}")

학습 데이터 준비 완료: 4890개


Loading weights: 100%|██████████| 290/290 [00:00<00:00, 1158.56it/s]


trainable params: 540,672 || all params: 494,573,440 || trainable%: 0.1093


Tokenizing train dataset: 100%|██████████| 4890/4890 [00:02<00:00, 1899.89 examples/s]
[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151645}.


0.5B 파인튜닝 시작...


Step,Training Loss
50,3.676462
100,1.932432
150,1.365853
200,1.289233
250,1.288663
300,1.241457
350,1.268182
400,1.194280
450,1.175845
500,1.135755


파인튜닝 완료!
저장 완료: /home/ubuntu/qwen_0.5b_ft/final


In [13]:
import os
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

# 파인튜닝된 0.5B 로드
ft_path = os.path.expanduser("~/qwen_0.5b_ft/final")
BASE_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer_05_ft = AutoTokenizer.from_pretrained(ft_path)
base_model_05 = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,
    device_map="auto"
)
model_05_ft = PeftModel.from_pretrained(base_model_05, ft_path)
model_05_ft.eval()
print("파인튜닝된 0.5B 로드 완료!")

# predict 함수
def predict_05_ft(text: str, max_new_tokens: int = 10) -> str:
    messages   = build_prompt(text)
    text_input = tokenizer_05_ft.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True,
    )
    input_ids = tokenizer_05_ft(text_input, return_tensors="pt").input_ids.to(model_05_ft.device)
    with torch.no_grad():
        output_ids = model_05_ft.generate(
            input_ids,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=1.0,
            pad_token_id=tokenizer_05_ft.eos_token_id,
        )
    new_tokens = output_ids[0][input_ids.shape[-1]:]
    raw_output = tokenizer_05_ft.decode(new_tokens, skip_special_tokens=True).strip().lower()
    for label in VALID_LABELS:
        if label in raw_output:
            return label
    return "invalid"

# dev 평가
all_results_05_ft = {}
for condition in CONDITIONS:
    print(f"\n{'='*55}")
    print(f"  조건: {condition}")
    print(f"{'='*55}")
    result  = run_condition(condition, dataset, predict_fn=predict_05_ft)
    metrics = evaluate(result)
    all_results_05_ft[condition] = {"raw": result, "metrics": metrics}
    pc = metrics["per_class"]
    print(f"  완료 → n={result['n']} | invalid={result['invalid_count']}")
    print(f"  macro-F1 : {metrics['macro_f1']:.4f}")
    print(f"  support={pc[0]:.3f}  deny={pc[1]:.3f}  query={pc[2]:.3f}  comment={pc[3]:.3f}")

# zero-shot vs 파인튜닝 비교
print("\n" + "="*75)
print(f"{'조건':<15} {'0.5B zero-shot':>15} {'0.5B finetuned':>15} {'차이':>8}")
print("="*75)
for condition in CONDITIONS:
    m_zs = all_results_05[condition]["metrics"]["macro_f1"]
    m_ft = all_results_05_ft[condition]["metrics"]["macro_f1"]
    print(f"{condition:<15} {m_zs:>15.3f} {m_ft:>15.3f} {m_ft-m_zs:>+8.3f}")
print("="*75)

Loading weights: 100%|██████████| 290/290 [00:00<00:00, 1174.13it/s]

파인튜닝된 0.5B 로드 완료!


NameError: name 'CONDITIONS' is not defined

In [14]:
import os
import json
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer
from sklearn.metrics import f1_score
from collections import Counter

# 변수 재정의
CONDITIONS = ["reply_only", "useful", "irrelevant", "conflicting", "mixed", "lexical"]
LABEL_MAP  = {"support": 0, "deny": 1, "query": 2, "comment": 3}
VALID_LABELS = {"support", "deny", "query", "comment"}

# 데이터 재로드
with open('context_conditions.json', encoding='utf-8') as f:
    full_dataset = json.load(f)
dataset = [d for d in full_dataset if d['split'] == 'dev']
print(f"dev 샘플 수: {len(dataset)}")

# evaluate 함수 재정의
def evaluate(result: dict) -> dict:
    golds = result["golds"]
    preds = result["preds"]
    if len(golds) == 0:
        return {"macro_f1": 0.0, "per_class": [0.0] * 4}
    macro_f1  = f1_score(golds, preds, average="macro", zero_division=0)
    per_class = f1_score(golds, preds, average=None, labels=[0,1,2,3], zero_division=0)
    return {"macro_f1": macro_f1, "per_class": per_class.tolist()}

# run_condition 함수 재정의
def run_condition(condition: str, data: list, predict_fn=None) -> dict:
    if predict_fn is None:
        predict_fn = predict
    golds, preds  = [], []
    invalid_count = 0
    skipped_count = 0
    total         = len(data)
    for i, d in enumerate(data):
        if d[condition] is None:
            skipped_count += 1
            continue
        pred = predict_fn(d[condition])
        if pred == "invalid":
            invalid_count += 1
            continue
        golds.append(LABEL_MAP[d["label"]])
        preds.append(LABEL_MAP[pred])
        if (i + 1) % 100 == 0:
            print(f"  [{condition}] {i+1}/{total} | 유효 {len(golds)} | invalid {invalid_count}")
    return {
        "golds":         golds,
        "preds":         preds,
        "invalid_count": invalid_count,
        "skipped_count": skipped_count,
        "n":             len(golds),
    }

# build_prompt 함수 재정의
SYSTEM_PROMPT = """You are a stance classification expert for social media discussions about rumours.

Classify the stance of the TARGET reply using exactly one of these four labels:

- support: The reply explicitly states the rumour IS true or confirmed.
- deny: The reply explicitly states the rumour IS false or fabricated.
- query: The reply asks for sources, evidence, or verification.
- comment: Everything else. The reply does not directly address whether the rumour is true or false.

Respond with ONLY one word: support, deny, query, or comment. No explanation."""

def build_prompt(text: str) -> list:
    cleaned = text
    cleaned = cleaned.replace("[Source]",     "Rumour post:")
    cleaned = cleaned.replace("[Context]",    "Previous reply:")
    cleaned = cleaned.replace("[Misleading]", "Another reply:")
    cleaned = cleaned.replace("[Target]",     "Reply to classify:")
    user_content = (
        f"Read the following and classify the stance of the 'Reply to classify'.\n\n"
        f"{cleaned}\n\n"
        f"Stance label (support / deny / query / comment):"
    )
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": user_content},
    ]

# 파인튜닝된 0.5B 로드
ft_path    = os.path.expanduser("~/qwen_0.5b_ft/final")
BASE_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer_05_ft = AutoTokenizer.from_pretrained(ft_path)
base_model_05   = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,
    device_map="auto"
)
model_05_ft = PeftModel.from_pretrained(base_model_05, ft_path)
model_05_ft.eval()
print("파인튜닝된 0.5B 로드 완료!")

# predict 함수
def predict_05_ft(text: str, max_new_tokens: int = 10) -> str:
    messages   = build_prompt(text)
    text_input = tokenizer_05_ft.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True,
    )
    input_ids = tokenizer_05_ft(text_input, return_tensors="pt").input_ids.to(model_05_ft.device)
    with torch.no_grad():
        output_ids = model_05_ft.generate(
            input_ids,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=1.0,
            pad_token_id=tokenizer_05_ft.eos_token_id,
        )
    new_tokens = output_ids[0][input_ids.shape[-1]:]
    raw_output = tokenizer_05_ft.decode(new_tokens, skip_special_tokens=True).strip().lower()
    for label in VALID_LABELS:
        if label in raw_output:
            return label
    return "invalid"

# dev 평가
all_results_05_ft = {}
for condition in CONDITIONS:
    print(f"\n{'='*55}")
    print(f"  조건: {condition}")
    print(f"{'='*55}")
    result  = run_condition(condition, dataset, predict_fn=predict_05_ft)
    metrics = evaluate(result)
    all_results_05_ft[condition] = {"raw": result, "metrics": metrics}
    pc = metrics["per_class"]
    print(f"  완료 → n={result['n']} | invalid={result['invalid_count']}")
    print(f"  macro-F1 : {metrics['macro_f1']:.4f}")
    print(f"  support={pc[0]:.3f}  deny={pc[1]:.3f}  query={pc[2]:.3f}  comment={pc[3]:.3f}")

dev 샘플 수: 1447


Loading weights: 100%|██████████| 290/290 [00:00<00:00, 1180.33it/s]
[transformers] The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


파인튜닝된 0.5B 로드 완료!

  조건: reply_only
  [reply_only] 100/1447 | 유효 100 | invalid 0
  [reply_only] 200/1447 | 유효 200 | invalid 0
  [reply_only] 300/1447 | 유효 300 | invalid 0
  [reply_only] 400/1447 | 유효 400 | invalid 0
  [reply_only] 500/1447 | 유효 500 | invalid 0
  [reply_only] 600/1447 | 유효 600 | invalid 0
  [reply_only] 700/1447 | 유효 700 | invalid 0
  [reply_only] 800/1447 | 유효 800 | invalid 0
  [reply_only] 900/1447 | 유효 900 | invalid 0
  [reply_only] 1000/1447 | 유효 1000 | invalid 0
  [reply_only] 1100/1447 | 유효 1100 | invalid 0
  [reply_only] 1200/1447 | 유효 1200 | invalid 0
  [reply_only] 1300/1447 | 유효 1300 | invalid 0
  [reply_only] 1400/1447 | 유효 1400 | invalid 0
  완료 → n=1447 | invalid=0
  macro-F1 : 0.3113
  support=0.000  deny=0.000  query=0.382  comment=0.863

  조건: useful
  [useful] 100/1447 | 유효 100 | invalid 0
  [useful] 200/1447 | 유효 200 | invalid 0
  [useful] 300/1447 | 유효 300 | invalid 0
  [useful] 400/1447 | 유효 400 | invalid 0
  [useful] 500/1447 | 유효 500 | invalid 0
  [

In [20]:
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer, SFTConfig
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch, os, gc

# 메모리 정리
torch.cuda.empty_cache()
gc.collect()

MODEL_NAME_15 = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer_15_ft = AutoTokenizer.from_pretrained(MODEL_NAME_15)
tokenizer_15_ft.pad_token = tokenizer_15_ft.eos_token

model_15_ft = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME_15,
    torch_dtype=torch.float16,
    device_map="auto"
)

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)
model_15_ft = get_peft_model(model_15_ft, lora_config)
model_15_ft.print_trainable_parameters()

sft_config = SFTConfig(
    output_dir=os.path.expanduser("~/qwen_1.5b_ft"),
    num_train_epochs=3,
    per_device_train_batch_size=2,        # ★ 8→2로 줄임
    gradient_accumulation_steps=8,        # ★ 2→8로 늘림 (실질 배치는 동일)
    learning_rate=2e-4,
    fp16=True,
    logging_steps=50,
    save_strategy="epoch",
    warmup_steps=100,
    lr_scheduler_type="cosine",
    report_to="none",
    gradient_checkpointing=True,          # ★ 메모리 절약
)

trainer = SFTTrainer(
    model=model_15_ft,
    args=sft_config,
    train_dataset=hf_dataset,
    processing_class=tokenizer_15_ft,
)

print("1.5B 파인튜닝 시작...")
trainer.train()
print("파인튜닝 완료!")

save_path = os.path.expanduser("~/qwen_1.5b_ft/final")
model_15_ft.save_pretrained(save_path)
tokenizer_15_ft.save_pretrained(save_path)
print(f"저장 완료: {save_path}")

Loading weights: 100%|██████████| 338/338 [00:00<00:00, 522.16it/s]


trainable params: 1,089,536 || all params: 1,544,803,840 || trainable%: 0.0705


Tokenizing train dataset: 100%|██████████| 4890/4890 [00:02<00:00, 1911.58 examples/s]
[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151645}.


1.5B 파인튜닝 시작...


Step,Training Loss
50,3.390912
100,1.822795
150,1.245701
200,1.178671
250,1.180401
300,1.141385
350,1.165350
400,1.101355
450,1.079328
500,1.045962


파인튜닝 완료!
저장 완료: /home/ubuntu/qwen_1.5b_ft/final


In [21]:
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch, os, gc

# 메모리 정리
torch.cuda.empty_cache()
gc.collect()

# 파인튜닝된 1.5B 로드
ft_path_15  = os.path.expanduser("~/qwen_1.5b_ft/final")
BASE_MODEL_15 = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer_15_ft = AutoTokenizer.from_pretrained(ft_path_15)
base_model_15   = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_15,
    torch_dtype=torch.float16,
    device_map="auto"
)
model_15_ft = PeftModel.from_pretrained(base_model_15, ft_path_15)
model_15_ft.eval()
print("파인튜닝된 1.5B 로드 완료!")

# predict 함수
def predict_15_ft(text: str, max_new_tokens: int = 10) -> str:
    messages   = build_prompt(text)
    text_input = tokenizer_15_ft.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True,
    )
    input_ids = tokenizer_15_ft(text_input, return_tensors="pt").input_ids.to(model_15_ft.device)
    with torch.no_grad():
        output_ids = model_15_ft.generate(
            input_ids,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=1.0,
            pad_token_id=tokenizer_15_ft.eos_token_id,
        )
    new_tokens = output_ids[0][input_ids.shape[-1]:]
    raw_output = tokenizer_15_ft.decode(new_tokens, skip_special_tokens=True).strip().lower()
    for label in VALID_LABELS:
        if label in raw_output:
            return label
    return "invalid"

# dev 평가
all_results_15_ft = {}
for condition in CONDITIONS:
    print(f"\n{'='*55}")
    print(f"  조건: {condition}")
    print(f"{'='*55}")
    result  = run_condition(condition, dataset, predict_fn=predict_15_ft)
    metrics = evaluate(result)
    all_results_15_ft[condition] = {"raw": result, "metrics": metrics}
    pc = metrics["per_class"]
    print(f"  완료 → n={result['n']} | invalid={result['invalid_count']}")
    print(f"  macro-F1 : {metrics['macro_f1']:.4f}")
    print(f"  support={pc[0]:.3f}  deny={pc[1]:.3f}  query={pc[2]:.3f}  comment={pc[3]:.3f}")

Loading weights: 100%|██████████| 338/338 [00:00<00:00, 512.99it/s]


파인튜닝된 1.5B 로드 완료!

  조건: reply_only
  [reply_only] 100/1447 | 유효 100 | invalid 0
  [reply_only] 200/1447 | 유효 200 | invalid 0
  [reply_only] 300/1447 | 유효 300 | invalid 0
  [reply_only] 400/1447 | 유효 400 | invalid 0
  [reply_only] 500/1447 | 유효 500 | invalid 0
  [reply_only] 600/1447 | 유효 600 | invalid 0
  [reply_only] 700/1447 | 유효 700 | invalid 0
  [reply_only] 800/1447 | 유효 800 | invalid 0
  [reply_only] 900/1447 | 유효 900 | invalid 0
  [reply_only] 1000/1447 | 유효 1000 | invalid 0
  [reply_only] 1100/1447 | 유효 1100 | invalid 0
  [reply_only] 1200/1447 | 유효 1200 | invalid 0
  [reply_only] 1300/1447 | 유효 1300 | invalid 0
  [reply_only] 1400/1447 | 유효 1400 | invalid 0
  완료 → n=1447 | invalid=0
  macro-F1 : 0.3432
  support=0.000  deny=0.112  query=0.371  comment=0.890

  조건: useful
  [useful] 100/1447 | 유효 100 | invalid 0
  [useful] 200/1447 | 유효 200 | invalid 0
  [useful] 300/1447 | 유효 300 | invalid 0
  [useful] 400/1447 | 유효 400 | invalid 0
  [useful] 500/1447 | 유효 500 | invalid 0
  [